In [1]:
pip install sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, torch, sys, warnings, pandas as pd

# CRITICAL: Set these BEFORE importing any other libraries
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Check Python and library versions
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

try:
    from transformers import __version__ as transformers_version
    print(f"Transformers version: {transformers_version}")
except ImportError as e:
    print(f"Error importing transformers: {e}")
    sys.exit(1)

from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    AutoModelForCausalLM, 
    AutoModelForSequenceClassification
)
from eco2ai import Tracker
warnings.filterwarnings("ignore")


C:\Users\User\anaconda3\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
C:\Users\User\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Python version: 3.11.4 | packaged by Anaconda, Inc. | (main, Jul  5 2023, 13:38:37) [MSC v.1916 64 bit (AMD64)]
PyTorch version: 2.8.0+cpu
Transformers version: 4.56.1


In [ ]:
# Suppress pandas deprecation warnings for cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)

# --- CONFIGURATION ---

# 1. Define the path where the emissions CSV will be saved.
CSV_FILE = "model_inference_emissions.csv"

# 2. Define the tasks and their corresponding prompts.
#    For tasks like NLI that need two sentences, use a dictionary.
PROMPT_LIBRARY = {
    "chain_of_thought_reasoning": "Solve step by step: A farmer has 150 meters of fencing to build a rectangular enclosure, and he wants to maximize the area he can enclose with it.",
    "translation_fr_en": "Translate this from French to English: 'Les lamas sont des créatures merveilleuses.'",
    "binary_sentiment_analysis": "I had a wonderful time at the new restaurant, the food was delicious!",
    "content_creation_story": "Write a short story that begins with this sentence: 'The old clock tower struck thirteen, and a chill ran down my spine.'",
    "nli_task": {
        "premise": "A man is playing a guitar on a stage.",
        "hypothesis": "A man is performing music for an audience."
    },
    "code_summarization_python": """
def fibonacci(n):
    \"\"\"This function generates the Fibonacci sequence up to n numbers.\"\"\"
    a, b = 0, 1
    result = []
    while len(result) < n:
        result.append(a)
        a, b = b, a + b
    return result
"""
}

# 3. Define your tasks and the list of models to test for each.

TASKS_AND_MODELS = {
    "chain_of_thought_reasoning": [
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\cot-flan-t5-base",
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\cot-flan-t5-small",
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\cot-t5-reasoning"
    ],
    
    "binary_sentiment_analysis": [
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\sentiment-analysis-distilroberta",
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\sentiment-analysis-distilbert",
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\sentiment-analysis-electra"
    ],
    "content_creation_story": [
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\content-creation-gpt2",
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\content-creation-eleutheraipythia-160m",
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\content-creation-distilgpt2"
    ],
    "nli_task": [
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\nli-flan-t5-base",
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\nli-deberta",
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\nli-electra"
    ],
    "code_summarization_python": [
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\code-summarization-tinycodellm",
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\code-summarization-codeparrot-small",
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\code_summarization-codet5-small"
    ],
    "translation_fr_en": [
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\translation-helsinki-nlpopus-mt-en", 
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\translation-helsinki-nlpopus-mt-en-fr", 
        r"C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\translation-t5-small" 
    ],
}

# --- CORE FUNCTION ---

def track_local_model_inference(task_name, prompt, model_path):
    """
    Tracks carbon emissions for a single model inference run, adapting to different model types.
    """
    if not os.path.isdir(model_path):
        print(f"Error: Model not found at {model_path}. Skipping.")
        return None

    tracker = Tracker(
        project_name=f"{task_name}-{os.path.basename(model_path)}",
        experiment_description="Local CPU-only inference", file_name=CSV_FILE, pue=1.0
    )

    result_text = "Inference Failed"
    try:
        tracker.start()

        tokenizer = AutoTokenizer.from_pretrained(model_path)

        # Gracefully try loading different model architectures
        try:
            model = AutoModelForSeq2SeqLM.from_pretrained(model_path, torch_dtype=torch.float32, device_map="cpu")
        except (ValueError, OSError):
            try:
                model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float32, device_map="cpu")
            except (ValueError, OSError):
                model = AutoModelForSequenceClassification.from_pretrained(model_path, torch_dtype=torch.float32, device_map="cpu")

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            if model.config.pad_token_id is None:
                model.config.pad_token_id = model.config.eos_token_id

        # Adapt tokenization for single vs. multi-part prompts
        if isinstance(prompt, dict):
            # For dict prompts like NLI: tokenizer(premise, hypothesis, ...)
            inputs = tokenizer(*prompt.values(), return_tensors="pt", truncation=True, max_length=512)
        else:
            # For standard string prompts
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

        with torch.no_grad():
            # Generative models (T5, GPT2, etc.) have a .generate() method
            if hasattr(model, "generate"):
                outputs = model.generate(**inputs, max_new_tokens=100, pad_token_id=tokenizer.pad_token_id)
                result_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            # Classification models (DistilBERT, DeBERTa, etc.) output logits
            else:
                logits = model(**inputs).logits
                pred_id = logits.argmax().item()
                # ✨ Convert predicted ID to a human-readable label
                if model.config.id2label:
                    result_text = f"Classification: {model.config.id2label[pred_id]}"
                else:
                    result_text = f"Classification ID: {pred_id}"

    except Exception as e:
        print(f"An error occurred during inference for {model_path}: {e}")
    finally:
        tracker.stop()

    # Read the latest run from the CSV
    if os.path.exists(CSV_FILE):
        df = pd.read_csv(CSV_FILE)
        if not df.empty:
            last_run = df.iloc[-1]
            return {
                "duration(s)": last_run.get('duration(s)'),
                "power_consumption(kWh)": last_run.get('power_consumption(kWh)'),
                "CO2_emissions(kg)": last_run.get('CO2_emissions(kg)'),
                "model_output": result_text
            }
    return None

# --- MAIN EXECUTION ---

if __name__ == "__main__":
    if os.path.exists(CSV_FILE):
        os.remove(CSV_FILE)

    all_results = []
    print("Starting model inference and carbon tracking with eco2ai...")
    
    for task_name, model_paths in TASKS_AND_MODELS.items():
        if task_name not in PROMPT_LIBRARY:
            print(f"Warning: Task '{task_name}' found but no prompt in PROMPT_LIBRARY. Skipping.")
            continue

        prompt = PROMPT_LIBRARY[task_name]
        print(f"\n----- Task: {task_name} -----")

        for model_path in model_paths:
            model_name = os.path.basename(model_path)
            print(f"  > Running model: {model_name}...")
            result = track_local_model_inference(task_name, prompt, model_path)

            if result:
                row_data = {
                    "Task": task_name, "Model": model_name,
                    "Duration (s)": f"{result['duration(s)']:.2f}",
                    "Energy (kWh)": f"{result['power_consumption(kWh)']:.8f}",
                    "CO2 Emissions (kg)": f"{result['CO2_emissions(kg)']:.8f}",
                    "Output": result['model_output']
                }
                all_results.append(row_data)
                print(f"  > Completed. CO2 Emitted: {result['CO2_emissions(kg)']:.8f} kg")
            else:
                print(f"  > Failed to get results for {model_name}.")

    # --- DISPLAY FINAL RESULTS TABLE ---
    if all_results:
        print("\n\n--- 📊 Final Emissions Summary (eco2ai) ---")
        results_df = pd.DataFrame(all_results)
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', 60)
        print(results_df.to_string())
    else:
        print("\n\n--- No results to display. Please check model paths and configurations. ---")

Starting model inference and carbon tracking with eco2ai...

----- Task: chain_of_thought_reasoning -----
  > Running model: cot-flan-t5-base...


`torch_dtype` is deprecated! Use `dtype` instead!


  > Completed. CO2 Emitted: 0.00000051 kg
  > Running model: cot-flan-t5-small...
  > Completed. CO2 Emitted: 0.00000155 kg
  > Running model: cot-t5-reasoning...
  > Completed. CO2 Emitted: 0.00000102 kg

----- Task: binary_sentiment_analysis -----
  > Running model: sentiment-analysis-distilroberta...


If you want to use `RobertaLMHeadModel` as a standalone, add `is_decoder=True.`
Some weights of RobertaForCausalLM were not initialized from the model checkpoint at C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\sentiment-analysis-distilroberta and are newly initialized: ['lm_head.bias', 'lm_head.decoder.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  > Completed. CO2 Emitted: 0.00006962 kg
  > Running model: sentiment-analysis-distilbert...
  > Completed. CO2 Emitted: 0.00000083 kg
  > Running model: sentiment-analysis-electra...
An error occurred during inference for C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\sentiment-analysis-electra: stat: path should be string, bytes, os.PathLike or integer, not NoneType
  > Completed. CO2 Emitted: 0.00000010 kg

----- Task: content_creation_story -----
  > Running model: content-creation-gpt2...
  > Completed. CO2 Emitted: 0.00000632 kg
  > Running model: content-creation-eleutheraipythia-160m...
  > Completed. CO2 Emitted: 0.00001046 kg
  > Running model: content-creation-distilgpt2...
  > Completed. CO2 Emitted: 0.00000366 kg

----- Task: nli_task -----
  > Running model: nli-flan-t5-base...
  > Completed. CO2 Emitted: 0.00000187 kg
  > Running model: nli-deberta...
  > Completed. CO2 Emitted: 0.00000137 kg
  > Running model: nli-electra...


If you want to use `ElectraForCausalLM` as a standalone, add `is_decoder=True.`
Some weights of ElectraForCausalLM were not initialized from the model checkpoint at C:\Users\User\Desktop\Plaksha\sem 4\ilgc 4\models\nli-electra and are newly initialized: ['generator_lm_head.bias', 'generator_predictions.LayerNorm.bias', 'generator_predictions.LayerNorm.weight', 'generator_predictions.dense.bias', 'generator_predictions.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  > Completed. CO2 Emitted: 0.00000151 kg

----- Task: code_summarization_python -----
  > Running model: code-summarization-tinycodellm...
  > Completed. CO2 Emitted: 0.00009725 kg
  > Running model: code-summarization-codeparrot-small...
  > Completed. CO2 Emitted: 0.00000546 kg
  > Running model: code_summarization-codet5-small...
  > Completed. CO2 Emitted: 0.00000560 kg

----- Task: translation_fr_en -----
  > Running model: translation-helsinki-nlpopus-mt-en...


In [2]:
import time
from openai import OpenAI

# Suppress warnings
warnings.filterwarnings("ignore")

API_KEY = "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"
API_BASE_URL = "https://openrouter.ai/api/v1"
CSV_FILE = "model_inference_emissions.csv"

# Define your API models
API_MODELS = {
    "Qwen": "qwen/qwen3-235b-a22b:free",
    "Mistral": "mistralai/mistral-7b-instruct",
    "DeepSeek": "deepseek/deepseek-r1-0528:free",
}

# Include the prompt library directly to avoid import issues
PROMPT_LIBRARY = {
    "chain_of_thought_reasoning": "Solve step by step: A farmer has 150 meters of fencing to build a rectangular enclosure, and he wants to maximize the area he can enclose with it.",
    "binary_sentiment_analysis": "Classify sentiment: 'The food was absolutely wonderful, from preparation to presentation, very pleasing and flavorful, and the service was attentive and friendly.' -> Is it positive or negative?",
    # Add other tasks if needed
}

def track_api_inference(task_name, model_name, model_id, prompt):
    tracker = Tracker(
        project_name=f"{task_name}-api",
        experiment_description=f"API inference with {model_name}",
        file_name=CSV_FILE,
        pue=1.0
    )

    client = OpenAI(base_url=API_BASE_URL, api_key=API_KEY)
    result = "API call failed."

    try:
        tracker.start()
        response = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150,
            temperature=0.7,
        )
        result = response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error: {e}")
        time.sleep(1)
    finally:
        tracker.stop()  # stops and writes CSV

    # Read CSV to get emissions data
    if os.path.exists(CSV_FILE):
        df = pd.read_csv(CSV_FILE)
        last = df.iloc[-1]  # latest run
        duration = last.get('duration(s)', None)
        energy = last.get('power_consumption(kWh)', None)
        co2 = last.get('CO2_emissions(kg)', None)
    else:
        duration = energy = co2 = None

    # Print results
    print(f"\n--- API Model Output for task '{task_name}' with model '{model_name}' ---")
    print(result)
    print(f"--- Estimated Carbon Emissions ---")
    if duration is not None:
        print(f"Duration: {duration} s")
        print(f"Energy consumed: {energy} kWh")
        print(f"Approximate CO2 emitted: {co2} kg")
    else:
        print("Could not retrieve emissions data from CSV.")
    print("-----------------------------------------\n")


if __name__ == "__main__":
    # Run only the tasks you want, e.g., TASK_NAME
    TASK_NAME = "chain_of_thought_reasoning"

    if TASK_NAME in PROMPT_LIBRARY:
        prompt = PROMPT_LIBRARY[TASK_NAME]
        for model_name, model_id in API_MODELS.items():
            track_api_inference(TASK_NAME, model_name, model_id, prompt)
    else:
        print(f"Error: Task '{TASK_NAME}' not found in PROMPT_LIBRARY")



--- API Model Output for task 'chain_of_thought_reasoning' with model 'Qwen' ---
<think>Okay, let's see. The problem is about a farmer who has 150 meters of fencing to build a rectangular enclosure and wants to maximize the area. Hmm, I need to solve this step by step.

First, I remember that for optimization problems involving rectangles and perimeter, calculus or completing the square might be useful. Let me start by recalling the formulas.

A rectangle has two lengths and two widths. Let's denote the length as L and the width as W. The perimeter P of a rectangle is given by P = 2L + 2W. The farmer has 150 meters of fencing, so the perimeter is 150 meters. So, 2L + 2W = 150
--- Estimated Carbon Emissions ---
Duration: 17.90469241142273 s
Energy consumed: 4.530063728959514e-07 kWh
Approximate CO2 emitted: 2.83388555711639e-07 kg
-----------------------------------------


--- API Model Output for task 'chain_of_thought_reasoning' with model 'Mistral' ---
<s> [OUT] To maximize the are

In [3]:


# --- CONFIGURATION ---

API_KEY = "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"
API_BASE_URL = "https://openrouter.ai/api/v1"
CSV_FILE = "api_inference_emissions.csv"

# Define your API models and map them to an estimation class
API_MODELS = {
    "Qwen-235B": {"id": "qwen/qwen3-235b-a22b:free", "estimation_class": "Large"},
    "Mistral-7B": {"id": "mistralai/mistral-7b-instruct", "estimation_class": "Small"},
    "DeepSeek-R1": {"id": "deepseek/deepseek-r1-0528:free", "estimation_class": "Large"},
}

# Estimates of CO2 emissions in kg per 1,000 generated tokens
# Based on figures from various research papers (e.g., Luccioni et al., 2023)
# These are approximations and will vary based on datacenter, hardware, etc.
MODEL_EMISSIONS_ESTIMATES = {
    # For models ~7B parameters
    "Small": 0.000021,
    # For very large models > 70B parameters
    "Large": 0.000250,
}

PROMPT_LIBRARY = {
    "chain_of_thought_reasoning": "Solve step by step: A farmer has 150 meters of fencing to build a rectangular enclosure, and he wants to maximize the area he can enclose with it.",
    "binary_sentiment_analysis": "Classify the sentiment of this sentence as positive or negative: 'The food was absolutely wonderful, from preparation to presentation, very pleasing and flavorful, and the service was attentive and friendly.'",
}

def track_api_inference(task_name, model_name, model_details):
    """
    Tracks local emissions and estimates remote emissions for an API call.
    Returns a dictionary with results.
    """
    tracker = Tracker(
        project_name=f"{task_name}-{model_name}-api",
        file_name=CSV_FILE, pue=1.0
    )
    client = OpenAI(base_url=API_BASE_URL, api_key=API_KEY)
    
    prompt = PROMPT_LIBRARY[task_name]
    model_id = model_details["id"]
    estimation_class = model_details["estimation_class"]
    
    api_result = "API call failed."
    local_co2 = 0.0
    estimated_remote_co2 = 0.0
    
    try:
        tracker.start()
        response = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=250,
            temperature=0.7,
        )
        api_result = response.choices[0].message.content.strip()
        
        # Get tokens generated to estimate remote emissions
        tokens_generated = response.usage.completion_tokens
        co2_per_1k_tokens = MODEL_EMISSIONS_ESTIMATES[estimation_class]
        estimated_remote_co2 = (tokens_generated / 1000) * co2_per_1k_tokens
        
    except Exception as e:
        print(f"  > Error during API call for {model_name}: {e}")
        time.sleep(1) # Wait a second before the next call on failure
    finally:
        tracker.stop()

    if os.path.exists(CSV_FILE):
        df = pd.read_csv(CSV_FILE)
        if not df.empty:
            local_co2 = df.iloc[-1].get('CO2_emissions(kg)', 0.0)

    return {
        "local_co2": local_co2,
        "estimated_remote_co2": estimated_remote_co2,
        "output": api_result
    }

if __name__ == "__main__":
    if os.path.exists(CSV_FILE):
        os.remove(CSV_FILE)

    all_results = []
    print("Starting API model inference and carbon tracking...")

    for task_name, prompt in PROMPT_LIBRARY.items():
        print(f"\n----- Task: {task_name} -----")
        for model_name, model_details in API_MODELS.items():
            print(f"  > Running model: {model_name}...")
            
            result = track_api_inference(task_name, model_name, model_details)
            
            if result:
                # The primary emission value to report is the estimated remote one
                total_co2 = result["estimated_remote_co2"]
                print(f"  > Completed. Estimated LLM CO2 Emitted: {total_co2:.8f} kg")
                
                row_data = {
                    "Task": task_name,
                    "Model": model_name,
                    "Local CO2 (kg)": f'{result["local_co2"]:.8f}',
                    "Estimated Remote CO2 (kg)": f'{result["estimated_remote_co2"]:.8f}',
                    "Output": result["output"]
                }
                all_results.append(row_data)

    # --- DISPLAY FINAL RESULTS TABLE ---
    if all_results:
        print("\n\n--- 📊 Final Emissions Summary ---")
        results_df = pd.DataFrame(all_results)
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', 60)
        print(results_df.to_string())

Starting API model inference and carbon tracking...

----- Task: chain_of_thought_reasoning -----
  > Running model: Qwen-235B...
  > Completed. Estimated LLM CO2 Emitted: 0.00006250 kg
  > Running model: Mistral-7B...
  > Completed. Estimated LLM CO2 Emitted: 0.00000525 kg
  > Running model: DeepSeek-R1...
  > Completed. Estimated LLM CO2 Emitted: 0.00006250 kg

----- Task: binary_sentiment_analysis -----
  > Running model: Qwen-235B...
  > Completed. Estimated LLM CO2 Emitted: 0.00006250 kg
  > Running model: Mistral-7B...
  > Completed. Estimated LLM CO2 Emitted: 0.00000006 kg
  > Running model: DeepSeek-R1...
  > Completed. Estimated LLM CO2 Emitted: 0.00006250 kg


--- 📊 Final Emissions Summary ---
                         Task        Model Local CO2 (kg) Estimated Remote CO2 (kg)                                                                                                                                                                                                            

In [ ]:
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

In [1]:
import os
import torch
import sys
import warnings
import pandas as pd
import gc  # Import the garbage collector

# CRITICAL: Set these BEFORE importing any other libraries
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Check Python and library versions
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

try:
    from transformers import __version__ as transformers_version
    print(f"Transformers version: {transformers_version}")
except ImportError as e:
    print(f"Error importing transformers: {e}")
    sys.exit(1)

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
)
from eco2ai import Tracker

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=FutureWarning)

# --- CONFIGURATION ---

# 1. Define the path where the emissions CSV will be saved.
CSV_FILE = "model_inference_emissions.csv"

# 2. Define the tasks and their corresponding prompts.
PROMPT_LIBRARY = {
    "translation_fr_en": "Translate this from French to English: 'Les lamas sont des créatures merveilleuses.'",
}

# 3. Define your tasks and the list of models to test for each.
#    ✨ Using model names from the Hugging Face Hub is the most reliable method.
#    The library will handle downloading and caching them correctly.
TASKS_AND_MODELS = {
    "translation_fr_en": [
        #"Helsinki-NLP/opus-mt-en-ROMANCE",  # The correct model for French-to-English
        #"Helsinki-NLP/opus-mt-en-fr",  # A model for English-to-French
        "t5-small",                    # A general-purpose sequence-to-sequence model
    ],
}

# --- CORE FUNCTION ---

def track_model_inference(task_name, prompt, model_name_or_path):
    """
    Tracks carbon emissions for a single model inference run, adapting to different model types.
    """
    tracker = Tracker(
        project_name=f"{task_name}-{os.path.basename(model_name_or_path)}",
        experiment_description="Local CPU-only inference",
        file_name=CSV_FILE,
        pue=1.0,
    )

    result_text = "Inference Failed"
    model = None
    tokenizer = None
    try:
        tracker.start()

        print(f"  > Loading tokenizer for '{model_name_or_path}'...")
        tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

        print(f"  > Loading model '{model_name_or_path}' (this may take a moment)...")
        # Gracefully try loading different model architectures
        try:
            model = AutoModelForSeq2SeqLM.from_pretrained(
                model_name_or_path, torch_dtype=torch.float32, device_map="cpu"
            )
        except (ValueError, OSError):
            try:
                model = AutoModelForCausalLM.from_pretrained(
                    model_name_or_path, torch_dtype=torch.float32, device_map="cpu"
                )
            except (ValueError, OSError):
                model = AutoModelForSequenceClassification.from_pretrained(
                    model_name_or_path, torch_dtype=torch.float32, device_map="cpu"
                )

        print("  > Model loaded. Preparing for inference...")
        if tokenizer.pad_token is None and hasattr(model.config, "eos_token_id"):
            tokenizer.pad_token = tokenizer.eos_token
            if model.config.pad_token_id is None:
                model.config.pad_token_id = model.config.eos_token_id

        # Adapt tokenization for single vs. multi-part prompts
        if isinstance(prompt, dict):
            inputs = tokenizer(
                *prompt.values(), return_tensors="pt", truncation=True, max_length=512
            )
        else:
            inputs = tokenizer(
                prompt, return_tensors="pt", truncation=True, max_length=512
            )

        with torch.no_grad():
            # Generative models (T5, MarianMT, etc.) have a .generate() method
            if hasattr(model, "generate"):
                outputs = model.generate(
                    **inputs, max_new_tokens=100, pad_token_id=tokenizer.pad_token_id
                )
                result_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            # Classification models output logits
            else:
                logits = model(**inputs).logits
                pred_id = logits.argmax().item()
                if model.config.id2label:
                    result_text = f"Classification: {model.config.id2label[pred_id]}"
                else:
                    result_text = f"Classification ID: {pred_id}"

    except Exception as e:
        print(f"An error occurred during inference for {model_name_or_path}: {e}")
    finally:
        tracker.stop()
        # ✨ Explicitly delete model and tokenizer and call garbage collector
        print("  > Cleaning up memory...")
        del model
        del tokenizer
        gc.collect()

    # Read the latest run from the CSV
    if os.path.exists(CSV_FILE):
        df = pd.read_csv(CSV_FILE)
        if not df.empty:
            last_run = df.iloc[-1]
            return {
                "duration(s)": last_run.get("duration(s)"),
                "power_consumption(kWh)": last_run.get("power_consumption(kWh)"),
                "CO2_emissions(kg)": last_run.get("CO2_emissions(kg)"),
                "model_output": result_text,
            }
    return None


# --- MAIN EXECUTION ---

if __name__ == "__main__":
    if os.path.exists(CSV_FILE):
        os.remove(CSV_FILE)

    all_results = []
    print("Starting model inference and carbon tracking with eco2ai...")

    for task_name, model_identifiers in TASKS_AND_MODELS.items():
        if task_name not in PROMPT_LIBRARY:
            print(
                f"Warning: Task '{task_name}' found but no prompt in PROMPT_LIBRARY. Skipping."
            )
            continue

        prompt = PROMPT_LIBRARY[task_name]
        print(f"\n----- Task: {task_name} -----")

        for model_id in model_identifiers:
            print(f"  > Running model: {model_id}...")
            result = track_model_inference(task_name, prompt, model_id)

            if result:
                row_data = {
                    "Task": task_name,
                    "Model": model_id,
                    "Duration (s)": f"{result['duration(s)']:.2f}",
                    "Energy (kWh)": f"{result['power_consumption(kWh)']:.8f}",
                    "CO2 Emissions (kg)": f"{result['CO2_emissions(kg)']:.8f}",
                    "Output": result["model_output"],
                }
                all_results.append(row_data)
                print(
                    f"  > Completed. CO2 Emitted: {result['CO2_emissions(kg)']:.8f} kg"
                )
            else:
                print(f"  > Failed to get results for {model_id}.")

    # --- DISPLAY FINAL RESULTS TABLE ---
    if all_results:
        print("\n\n--- 📊 Final Emissions Summary (eco2ai) ---")
        results_df = pd.DataFrame(all_results)
        pd.set_option("display.max_rows", None)
        pd.set_option("display.max_columns", None)
        pd.set_option("display.width", None)
        pd.set_option("display.max_colwidth", 60)
        print(results_df.to_string())
    else:
        print(
            "\n\n--- No results to display. Please check model names and configurations. ---"
        )

C:\Users\User\anaconda3\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
C:\Users\User\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Python version: 3.11.4 | packaged by Anaconda, Inc. | (main, Jul  5 2023, 13:38:37) [MSC v.1916 64 bit (AMD64)]
PyTorch version: 2.8.0+cpu


KeyboardInterrupt: 